In [1]:
from spatial_guidance import PipelineConfig, StrayDatasetConfig, StrayScannerDataParserConfig, StrayScannerPaths
from utils import CONSOLE

In [4]:
pipeline_config = PipelineConfig(
    stack="airflow_stack",
    # stack="default",
    dataset_config=StrayDatasetConfig(
        data_parser_config=StrayScannerDataParserConfig(
            paths=StrayScannerPaths(
                dataset_dir="SmartAIs Recorded Data/baustelle"  # relative to root/.data
            )
        ),
        is_rotated=True,
        verbose=False,
    ),
    enable_artifact_metadata_global=False,
)
pipeline = pipeline_config.setup_target()
# pipeline_config.inspect()

Propagated verbose=True from PipelineConfig to StrayDatasetConfig

Activated ZenML stack: airflow_stack

Initialized Gemini detector with model: gemini-2.0-flash

Initialized SceneVisualizer with debug=False

In [5]:
ret = pipeline(420, None)

Initiating a new run for the pipeline: SpatialUnderstandingPipeline.
Unable to find a build to reuse. A previous build can be reused when the following conditions are met:
  * The existing build was created for the same stack, ZenML version and Python version
  * The stack contains a container registry
  * The Docker settings of the pipeline and all its steps are the same as for the existing build.
Building Docker image(s) for pipeline SpatialUnderstandingPipeline.
Building Docker image zenml:SpatialUnderstandingPipeline-orchestrator.
Step 1/7 : FROM zenmldocker/zenml:0.80.1-py3.11
Step 2/7 : WORKDIR /app
Step 3/7 : ENV ZENML_LOGGING_COLORS_DISABLED=False
Step 4/7 : ENV ZENML_ENABLE_REPO_INIT_WARNINGS=False
Step 5/7 : ENV ZENML_CONFIG_PATH=/app/.zenconfig
Step 6/7 : COPY . .
Step 7/7 : RUN chmod -R a+rw .
Finished building Docker image(s).
Archiving pipeline code directory: /Users/jd/Desktop/repos/dlvc-04-spatial-understanding/notebooks. If this is taking longer than you expected, make

In [7]:
ret

PipelineRunResponse(body=PipelineRunResponseBody(created=datetime.datetime(2025, 4, 15, 13, 13, 35, 460533), updated=datetime.datetime(2025, 4, 15, 13, 13, 35, 460537), user=UserResponse(body=UserResponseBody(created=datetime.datetime(2025, 4, 8, 19, 26, 38, 351374), updated=datetime.datetime(2025, 4, 8, 19, 33, 45, 196633), active=True, activation_token=None, full_name='Jan Duchscherer', email_opted_in=False, is_service_account=False, is_admin=True, default_project_id=None), metadata=None, resources=None, id=UUID('f39ce7a5-9179-4d26-8202-2c47aed87431'), permission_denied=False, name='default'), status=<ExecutionStatus.INITIALIZING: 'initializing'>, stack=StackResponse(body=StackResponseBody(created=datetime.datetime(2025, 4, 8, 19, 29, 44, 68472), updated=datetime.datetime(2025, 4, 8, 19, 29, 44, 68480), user=UserResponse(body=UserResponseBody(created=datetime.datetime(2025, 4, 8, 19, 26, 38, 351374), updated=datetime.datetime(2025, 4, 8, 19, 33, 45, 196633), active=True, activation_t

In [9]:
ret.id

UUID('b5d8daef-673d-4ed0-9b68-345268eff4d3')

In [ ]:
from time import sleep
from zenml.client import Client

def wait_for_run_completion(run_id, poll_interval=10, timeout=1800):
    """
    Wait for a ZenML pipeline run to finish and return the refreshed run object.
    """
    client = Client()
    waited = 0
    while True:
        run = client.get_pipeline_run(run_id)
        status = run.status.value if hasattr(run, "status") else run.status
        print(f"Pipeline run status: {status}")
        if status in ("completed", "failed", "canceled"):
            break
        sleep(poll_interval)
        waited += poll_interval
        if waited > timeout:
            raise TimeoutError("Pipeline run did not finish in time.")
    return run

# Usage in your notebook:
# ret = pipeline(...)
run = wait_for_run_completion('b5d8daef-673d-4ed0-9b68-345268eff4d3')
print(run.steps)  # Now should be populated

Pipeline run status: initializing
Pipeline run status: initializing
Pipeline run status: initializing
Pipeline run status: initializing
Pipeline run status: initializing
Pipeline run status: initializing


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:24                                                                                   │
│                                                                                                  │
│   21                                                                                             │
│   22 # Usage in your notebook:                                                                   │
│   23 # ret = pipeline(...)                                                                       │
│ ❱ 24 run = wait_for_run_completion(ret.id)                                                       │
│   25 print(run.steps)  # Now should be populated                                                 │
│   26                                                                                             │
│                                                                                                  │
│ in wait_for_run_completion:16                                                                    │
│                                                                                                  │
│   13 │   │   print(f"Pipeline run status: {status}")                                             │
│   14 │   │   if status in ("completed", "failed", "canceled"):                                   │
│   15 │   │   │   break                                                                           │
│ ❱ 16 │   │   sleep(poll_interval)                                                                │
│   17 │   │   waited += poll_interval                                                             │
│   18 │   │   if waited > timeout:                                                                │
│   19 │   │   │   raise TimeoutError("Pipeline run did not finish in time.")                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt